# പാഠം 11 - ਏജന്റ്-ടു-ഏജന്റ് (A2A) പ്രോട്ടോക്കോൾ


## സജ്ജീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A പ്രോട്ടോക്കോൾ എന്താണ്?

The **എജന്റ്-ടു-എജന്റ് (A2A) പ്രോട്ടോക്കോൾ** ഒരു തുറന്ന സ്‌റ്റാൻഡേർഡാണ്, അത് AI ഏജന്റുകൾക്ക് ആശയവിനിമയം നടത്താനും, പരസ്പരം കണ്ടെത്താനും, സഹകരിക്കാനും അനുവദിക്കുന്നു — അവ വ്യത്യസ്ത ഫ്രെയിംവർക്കുകളിൽ നിർമ്മിക്കപ്പെട്ടിരിക്കുമ്പോഴും അല്ലെങ്കിൽ വ്യത്യസ്ത സേവനങ്ങളിലൂടെ ഹോസ്റ്റ് ചെയ്തിരിക്കുമ്പോഴും.

Key concepts:

- **കണ്ടെത്തൽ** – ഏജന്റുകൾ അവരുടെ കഴിവുകൾ വിവരിക്കുന്ന ഒരു *ഏജന്റ് കാർഡ്* പ്രസിദ്ധീകരിക്കുന്നു, അതിലൂടെ മറ്റുള്ള ഏജന്റുകൾ (അഥവാ ഓർക്കസ്ട്രേറ്റർമാർ) ഒരു ടാസ്‌കിന് അനുയോജ്യമായ വിദഗ്ധനെ കണ്ടെത്താൻ എളുപ്പമാകുന്നു.
- **സന്ദേശം കൈമാറ്റം** – ഏജന്റുകൾ ഒരു പൊതുവായ പ്രോട്ടോക്കോളിലൂടെ ഘടനയുള്ള സന്ദേശങ്ങൾ കൈമാറുന്നു, അതിലൂടെ ഒരു ഏജന്റിന്റെ അഭ്യർത്ഥന അതിന്റെ ആഭ്യന്തര നടപ്പിലാക്കലിനെ ആശ്രയിക്കാതെ മറ്റൊരു ഏജന്റിനാൽ മനസ്സിലാക്കി പാലിക്കപ്പെടാൻ കഴിയും.
- **ടാസ്‌കിന്റെ ജീവിത ചക്രം** – A2A *സമർപ്പിച്ചു*, *പ്രവർത്തനത്തിലാണ്*, *പൂർത്തിയായി*, and *പരാജയപ്പെട്ടു* പോലുള്ള നിലകൾ നിർവചിക്കുന്നു, അതിലൂടെ ഓർക്കസ്ട്രേറ്ററിന് ഏൽപ്പിച്ച ടാസ്‌കിന്റെ പുരോഗതി എങ്ങനെ നടക്കുന്നുവെന്നത് പൂർണ്ണമായി ദൃശ്യമാക്കുന്നു.

ഈ പാഠത്തിൽ നാം A2A-ശൈലിയിലെ സഹകരണത്തെ അനുകരിക്കുകയാണ് — മൂന്ന് വിദഗ്ധ യാത്രാ ഏജന്റുകളെ ഒരു വർക്ക്‌ഫ്ലോയിൽ ബന്ധിപ്പിച്ച്, ഓരോ ഏജന്റും തന്റെ വിദഗ്ധത സംഭാവന ചെയ്യുകയും ഫലങ്ങൾ അടുത്തവന് കൈമാറുകയും ചെയ്യുന്നു.


## വിശിഷ്ടമായ യാത്രാ ഏജന്റുകൾ സൃഷ്ടിക്കൽ


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## വർക്ക്ഫ്ലോ മുഖേന മൾട്ടി-എജന്റ് സഹകരണം

A2A സന്ദേശമാറ്റം പ്രതിഫലിപ്പിക്കുന്ന ക്രമം അനുസരിക്കുന്ന വർക്ക്ഫ്ലോയിലേക്ക് നാം ആ മൂന്ന് ഏജന്റുകളെയും ബന്ധിപ്പിക്കുന്നു:

1. **CurrencyExchangeAgent** ഉപയോക്താവിന്റെ അഭ്യർത്ഥന സ്വീകരിച്ച് നാണയ സംബന്ധമായ മാർഗ്ഗനിർദ്ദേശങ്ങൾ നൽകുന്നു.
2. **ActivityPlannerAgent** സമൃദ്ധമായ പശ്ചാത്തലം സ്വീകരിച്ച് പ്രവർത്തന ശുപാർശകൾ ചേർക്കുന്നു.
3. **TravelManagerAgent** ഇരുവിധ ഇൻപുട്ടുകളും സംശ്ലേഷിച്ച് അന്തിമ യാത്രാ സംഗ്രഹമായി രൂപപ്പെടുത്തുന്നു.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ഉത്പാദന പരിസ്ഥിതിയിൽ A2A മനസ്സിലാക്കൽ

ഉത്പാദന പരിസ്ഥിതിയിൽ A2A പ്രോട്ടോക്കോൾ സേവനങ്ങൾക്കിടയിലെ ശക്തമായ സാഹചര്യങ്ങൾ സാദ്ധ്യമാക്കുന്നു:

| ശേഷി | വിവരണം |
|---|---|
| **ഫ്രെയിംവർക്കുകൾക്കിടയിലെ ഇന്റർഓപ്പറബിലിറ്റി** | ഒരു ഫ്രെയിംവർക്കിൽ നിർമ്മിച്ച ഏജന്റ് മറ്റ് ഏതെങ്കിലും A2A-അനുകൂല ഫ്രെയിംവർക്കിൽ നിർമ്മിച്ച ഏജന്റിനോട് ചുമതലകൾ നിയോഗിക്കാൻ കഴിയും, ഇത് സംഘടനകളിലുടനീളം യഥാർത്ഥ ഇന്റർഓപ്പറബിലിറ്റിയെ അനുവദിക്കുന്നു. |
| **സേവന പരിധികൾ** | ഏജന്റുകൾ വേർതിരിച്ച മൈക്രോസർവിസുകളിൽ, ക്ലൗഡ് റീജ്യന്റുകളിൽ, അല്ലെങ്കിൽ വ്യത്യസ്ത സംഘടനകളിൽ താമസിച്ചിരുന്നാലും സഹജമായി സഹകരിക്കാനാകും. |
| **ഡൈനാമിക് കണ്ടെത്തൽ** | ഒരു ഓർക്കസ്ട്രേറ്റർ റൺടൈമിൽ Agent Card രജിസ്ട്രിയെ ചോദ്യം ചെയ്ത് നിശ്ചിത ഉപപ്രവൃതിക്ക് ഏറ്റവും അനുയോജ്യമായ ειδഗ്ധനെ കണ്ടെത്താം. |
| **സ്റ്റ്രീമിംഗ് & പുഷ് നോട്ടിഫിക്കേഷനുകൾ** | A2A Server-Sent Events (SSE) ഉപയോഗിച്ച് യഥാർത്ഥ-സമയ പുരോഗതി അപ്ഡേറ്റുകൾക്കും ദൈർഘ്യമേറിയ タസ്കുകൾക്കുള്ള പുഷ് അറിയിപ്പുകൾക്കും പിന്തുണ നൽകുന്നു. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. **A2A പ്രോട്ടോക്കോൾ എന്താണ്** — ഏജന്റ്-ടു-ഏജന്റ് കണ്ടെത്തൽ, സന്ദേശവിനിമയം,
   ഒപ്പം ടാസ്‌ക് മാനേജ്മെന്റ് എന്നിവയ്ക്കുള്ള ഒരു തുറന്ന സ്റ്റാൻഡാർഡ്.
2. **പ്രത്യേക ഏജന്റുകൾ എങ്ങനെ സൃഷ്ടിക്കാം** — ഒരു Currency Exchange ഏജന്റ്, ഒരു Activity Planner ഏജന്റ്, এবং ഒരു Travel Manager ഓർക്കസ്ട്രേറ്റർ.
3. **ഏജന്റുകൾ ഒരു workflow-ലിൽ എങ്ങനെ ബന്ധിപ്പിക്കാമെന്ന്** — `WorkflowBuilder` ഉപയോഗിച്ച് എജന്റുകൾക്കിടയിലെ ക്രമപരമായ
   സന്ദേശ കൈമാറ്റം മോഡൽ ചെയ്യാൻ.
4. **A2A പ്രൊഡക്ഷനിൽ എങ്ങനെ പ്രവർത്തിക്കുന്നു** — ക്രോസ്-ഫ്രെയിംവർക്ക്, ക്രോസ്-സർവീസ് സഹകരണത്തെ സജ്ജമാക്കുകയും
   ഡൈനാമിക് കണ്ടെത്തലും സ്ട്രീമിംഗ് അപ്‌ഡേറ്റുകളും പ്രദാനം ചെയ്യുകയും ചെയ്യുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ഡിസ്‌ക്ലെയിമർ:
ഈ രേഖ AI അധിഷ്ഠിത പരിഭാഷാ സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി പരിശ്രമിച്ചുവെങ്കിലും, ഓട്ടോമേറ്റഡ് വിവർത്തനങ്ങളിൽ ചില തെറ്റുകളും അസിസ്റ്റിരതകളും ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. യഥാർത്ഥ രേഖയുടെ സ്വദേശഭാഷയിലാണ് ഔദ്യോഗികമായ ഉറവിടമെന്ന നിലയിൽ കണക്കാക്കേണ്ടത്. അതീവഗൗരവമുള്ള വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മാനവ പരിഭാഷ നിർദേശിക്കുന്നു. ഈ വിവർത്തനത്തിന്റെ ഉപയോഗത്തിൽ നിന്നുണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണങ്ങളോ തെറ്റായ വ്യാഖ്യാനങ്ങളോ സംബന്ധിച്ച് ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
